# 08 — Advanced Machine Learning Forecasting Models
**Random Forest and Gradient Boosted Decision Trees (GBDT)**

Evaluates non-linear ensemble models for groundwater depth forecasting.

In [ ]:
import sys, os, pandas as pd, numpy as np, matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
sys.path.insert(0, os.path.abspath('..'))
from pipelines.preprocess_data import run_preprocessing_pipeline
from src.evaluation import evaluate_regression

train_df, test_df = run_preprocessing_pipeline(sample_mode=False)
feature_cols = [
    'month_sin', 'month_cos', 'doy_sin', 'doy_cos',
    'depth_to_water_mbgl_lag_1',
    'depth_to_water_mbgl_roll_mean_3',
    'specific_yield_est',
    'recharge_potential_index',
    'stage_of_extraction_pct'
]

mean_depth = float(train_df['depth_to_water_mbgl'].dropna().mean() or 12.0)
for col in ['depth_to_water_mbgl_lag_1', 'depth_to_water_mbgl_roll_mean_3']:
    train_df[col] = train_df[col].fillna(mean_depth)
    test_df[col] = test_df[col].fillna(mean_depth)
for col in ['specific_yield_est', 'recharge_potential_index', 'stage_of_extraction_pct']:
    train_df[col] = train_df[col].fillna(train_df[col].median())
    test_df[col] = test_df[col].fillna(train_df[col].median())

rf = RandomForestRegressor(n_estimators=100, max_depth=7, random_state=42)
rf.fit(train_df[feature_cols].values, train_df['depth_to_water_mbgl'].values)
preds = rf.predict(test_df[feature_cols].values)

metrics = evaluate_regression(test_df['depth_to_water_mbgl'].values, preds)
print('Random Forest Performance on Test Split:')
display(pd.DataFrame([metrics]))

# Feature importances
fi = pd.Series(rf.feature_importances_, index=feature_cols).sort_values(ascending=False)
plt.figure(figsize=(8, 4))
fi.plot(kind='barh', color='#0ea5e9')
plt.title('Random Forest Feature Importances')
plt.xlabel('Gini Importance')
plt.gca().invert_yaxis()
plt.show()